# Lab: CI/CD Automation for Machine Learning with GitHub Actions

## Objective
In this lab, you will build a production-grade CI/CD pipeline. In an enterprise MLOps setting, GitHub Actions is the **orchestrator** (the control plane), not the heavy compute node. 

Our pipeline will:
1. Ensure **Code Quality** using static analysis (`flake8`).
2. Run **Automated Tests** on our data processing logic (`pytest`).
3. Optimize execution speed using **Dependency Caching**.
4. Verify data lineage and track metrics using **DVC** and **GitHub Step Summaries**.

---

## 1. Code Quality & Linting (Static Analysis)

Before we even think about training a model, we must ensure our code is syntactically correct and follows team standards.

We have included a file `src/bad_code.py` that contains deliberate formatting and syntax issues. Let's run the linter locally first to see what CI will catch.

In [7]:
!flake8 src/bad_code.py

src/bad_code.py:3:5: F401 'math' imported but unused


*(Notice the errors? Don't fix them yet! We want to see GitHub Actions catch them in the cloud.)*

### Automated Testing (Pytest)
Similarly, we have a simple unit test in `tests/test_prepare.py`. Let's verify it runs locally:

In [9]:
!pytest tests/

============================= test session starts =============================
platform win32 -- Python 3.12.12, pytest-9.0.3, pluggy-1.6.0
rootdir: c:\Users\Nitvn\Documentos\Dev\lab_automation
configfile: pyproject.toml
plugins: anyio-4.13.0, Faker-40.19.1, hydra-core-1.3.2
collected 0 items / 1 error

=================================== ERRORS ====================================
___________________ ERROR collecting tests/test_prepare.py ____________________
tests\test_prepare.py:23: in <module>
    test_prepare_data_split_ratio()
tests\test_prepare.py:21: in test_prepare_data_split_ratio
    assert len(test) == 20
E   assert 21 == 20
E    +  where 21 = len(     feature1  target\n80         80      80\n81         81      81\n82         82      82\n83         83      83\n84      ...5      95\n96         96      96\n97         97      97\n98         98      98\n99         99      99\n100       100     100)
------------------------------- Captured stdout -------------------------------

**Create the GitHub Actions Workflow Directory:**

In [10]:
%%bash
!mkdir -p .github/workflows

<3>WSL (8167 - Relay) ERROR: CreateProcessCommon:818: execvpe(/bin/bash) failed: No such file or directory


CalledProcessError: Command 'b'!mkdir -p .github/workflows\n'' returned non-zero exit status 1.

---

## 2. Defining the CI Workflow

Create a new file named `ml_ci.yml` inside `.github/workflows/`. We will build this file covering all 4 principles.

**Task 1: GitHub Actions Workflow**
**GRADED TASK: Define ml_ci.yml**

```yaml
%%writefile .github/workflows/ml_ci.yml
name: MLOps CI Pipeline

# TODO: Define 'on' triggers for push and pull_request on 'main'

jobs:
  validate-and-report:
    # TODO: Run on ubuntu-latest

    steps:
    - name: Checkout Repository
      uses: actions/checkout@v4

    # TODO: Setup Python 3.10 with 'pip' caching
    
    # TODO: Install dependencies (requirements.txt, flake8, pytest)

    # TODO: Run flake8 on src/
    
    # TODO: Run pytest on tests/
    
    # TODO: Run dvc pull (or echo fallback)
    
    # TODO: Generate Markdown report using $GITHUB_STEP_SUMMARY
```

**Verify Task 1:**
Run the following Python cell to use the autograder.

In [ ]:
import unittests
unittests.test_github_actions_workflow()

---

## 3. The "Fix the Build" Exercise

Now it's time to test the CI pipeline in the cloud. Run the cell below to push your code to GitHub.

In [ ]:
!git add .github/workflows/ml_ci.yml src/bad_code.py tests/test_prepare.py
!git commit -m "Add enterprise CI pipeline with tests and linting"
!git push origin main

**Out-of-Notebook Instructions:**
1. Open your browser and navigate to your GitHub repository.
2. Click on the **Actions** tab.
3. You will see the build **fail** at the "Run Linter" step because of `bad_code.py`.

### Fixing the Error

Now, let's fix the broken code using the `%%writefile` magic command, and push the fix!

In [ ]:
%%writefile src/bad_code.py
def calculate_accuracy(y_true, y_pred):
    correct = sum(1 for true, pred in zip(y_true, y_pred) if true == pred)
    return correct / len(y_true)

Verify it passes locally now:

In [ ]:
!flake8 src/bad_code.py

Push the fix:

In [ ]:
!git add src/bad_code.py
!git commit -m "Fix flake8 formatting errors"
!git push origin main

**Final Check:**
Go back to the **Actions** tab on GitHub. Watch the pipeline turn **green**! Finally, click into the successful run and check the **Summary** tab to see the Markdown report generated by `$GITHUB_STEP_SUMMARY`.

---

## Conclusion
You have now implemented a mature CI pipeline. You have utilized caching for speed, linting for code quality, pytest for logic validation, and GitHub step summaries for reporting. You also understand why the CI runner fetches data (`dvc pull`) but delegates heavy model training to external compute platforms!